# Bronze to Silver for Frost station sources

The purpose is to create a reliable reference table of Norwegian Frost stations for spatially joining temperature observations to geographic areas.

Keep stations with usable identifiers and coordinates. Quarantine invalid records with a reason. Retain station names and metadata.

In [1]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession, functions as F
from silver_layer.frost_sources import ingest

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"

spark = (
    SparkSession.builder.appName("Frost Sources Silver")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

print("Project:", PROJECT_ROOT)

26/11/20 10:12:25 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/20 10:12:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 10:12:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 10:12:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Project: /home/roland/UiS/rewrite/group14


## Ingest

In [3]:
result = ingest(
    bronze_root=BRONZE_ROOT,
    silver_root=SILVER_ROOT,
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)
result

{
  "dataset": "frost_sources",
  "status": "success",
  "rows": 1589,
  "quarantined_rows": 24,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/frost/stations",
  "quarantine_path": "/home/roland/UiS/rewrite/group14/data/silver/quarantine/frost_sources"
}


{'dataset': 'frost_sources',
 'status': 'success',
 'rows': 1589,
 'quarantined_rows': 24,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/frost/stations',
 'quarantine_path': '/home/roland/UiS/rewrite/group14/data/silver/quarantine/frost_sources'}

## Inspect Silver

In [4]:
stations = spark.read.parquet(result["silver_path"])
quarantine = spark.read.parquet(result["quarantine_path"])

print(f"Stations: {stations.count():,}")
print(f"Quarantined: {quarantine.count():,}")

stations.printSchema()
stations.orderBy("station_id").show(20, truncate=False)

Stations: 1,589
Quarantined: 24
root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- municipality_code: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- county_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- elevation_m: double (nullable = true)
 |-- valid_from: string (nullable = true)
 |-- valid_to: string (nullable = true)

+----------+-----------------------+-----------------+-----------------+-----------------+-----------+-----------+-------+----------+----------+-----------+------------------------+--------+
|station_id|name                   |short_name       |municipality_code|municipality_name|county_code|county_name|country|latitude  |longitude |elevation_m|valid_from              |valid_to|
+----------+-------------------

In [5]:
print("Quarantine reasons:")
quarantine.groupBy("quarantine_reason").count().orderBy(F.desc("count")).show(
    truncate=False
)
print("Quarantine examples:")

quarantine.show(15, truncate=False)

Quarantine reasons:
+-------------------+-----+
|quarantine_reason  |count|
+-------------------+-----+
|INVALID_COORDINATES|24   |
+-------------------+-----+

Quarantine examples:
+----------+-----------------+-----------------+-----------------+-------------------+-----------+-------------------+-------+--------+---------+-----------+------------------------+--------+-------------------+-----------------------------------------------------------------------------------+
|station_id|name             |short_name       |municipality_code|municipality_name  |county_code|county_name        |country|latitude|longitude|elevation_m|valid_from              |valid_to|quarantine_reason  |_source_file                                                                       |
+----------+-----------------+-----------------+-----------------+-------------------+-----------+-------------------+-------+--------+---------+-----------+------------------------+--------+-------------------+---------------

In [6]:
print(
    "Duplicate station IDs:",
    stations.groupBy("station_id").count().where("count > 1").count(),
)
print(
    "Null coordinates:", stations.where("latitude IS NULL OR longitude IS NULL").count()
)

print("Stations by municipality (top 15):")
stations.groupBy("municipality_code", "municipality_name").count().orderBy(
    F.desc("count")
).show(15, truncate=False)

print("Country codes:")
stations.groupBy("country").count().show()

Duplicate station IDs: 0
Null coordinates: 0
Stations by municipality (top 15):
+-----------------+-----------------+-----+
|municipality_code|municipality_name|count|
+-----------------+-----------------+-----+
|2600             |NORDSJØEN        |38   |
|301              |OSLO             |36   |
|NULL             |NULL             |30   |
|3301             |DRAMMEN          |22   |
|4640             |SOGNDAL          |20   |
|4618             |ULLENSVANG       |20   |
|4601             |BERGEN           |20   |
|5501             |TROMSØ           |20   |
|4003             |SKIEN            |18   |
|5001             |TRONDHEIM        |17   |
|1508             |ÅLESUND          |17   |
|3905             |TØNSBERG         |16   |
|4647             |SUNNFJORD        |16   |
|3201             |BÆRUM            |15   |
|1806             |NARVIK           |15   |
+-----------------+-----------------+-----+
only showing top 15 rows

Country codes:
+-------+-----+
|country|count|
+-------+--

## Frost temperature observations

In [7]:
observations_path = SILVER_ROOT / "frost" / "observations"

if observations_path.exists():
    observations = spark.read.parquet(str(observations_path))
    missing = observations.join(
        stations.select("station_id"), "station_id", "left_anti"
    )
    total_observations = observations.count()
    unmatched_observations = missing.count()
    matched_observations = total_observations - unmatched_observations
    missing_station_count = missing.select("station_id").distinct().count()

    print(f"Total Silver observations: {total_observations:,}")
    print(
        f"With valid station metadata: {matched_observations:,} ({matched_observations / total_observations:.2%})"
    )
    print(
        f"Without valid station metadata: {unmatched_observations:,} ({unmatched_observations / total_observations:.2%})"
    )
    print(f"Distinct unmatched station IDs: {missing_station_count:,}")
    print("\nTop unmatched stations by observation count:")
    missing.groupBy("station_id").count().orderBy(F.desc("count")).show(
        20, truncate=False
    )

    print("\nUnmatched stations by metadata quarantine reason:")
    missing.select("station_id").distinct().join(
        quarantine.select("station_id", "quarantine_reason").distinct(),
        "station_id",
        "left",
    ).groupBy("quarantine_reason").count().show(truncate=False)
else:
    print("No observations found in Silver layer.")

Total Silver observations: 76,715,902
With valid station metadata: 73,672,565 (96.03%)
Without valid station metadata: 3,043,337 (3.97%)
Distinct unmatched station IDs: 13

Top unmatched stations by observation count:


+----------+------+
|station_id|count |
+----------+------+
|SN77037   |984078|
|SN77042   |956126|
|SN77051   |418625|
|SN77046   |300114|
|SN77001   |159103|
|SN77055   |63610 |
|SN77048   |46628 |
|SN77049   |38620 |
|SN77035   |25212 |
|SN77043   |23156 |
|SN77032   |14995 |
|SN77005   |10501 |
|SN77006   |2569  |
+----------+------+


Unmatched stations by metadata quarantine reason:


+-------------------+-----+
|quarantine_reason  |count|
+-------------------+-----+
|INVALID_COORDINATES|13   |
+-------------------+-----+



# Inspect all offshore regions

In [8]:
offshore = ["NORDSJØEN", "NORSKEHAVET", "BARENTSHAVET", "KONTINENTALSOKKELEN"]

print("Offshore regions:")
stations.filter(F.col("county_name").isin(offshore)).select(
    "station_id",
    "name",
    "county_name",
    "municipality_name",
    "municipality_code",
    "latitude",
    "longitude",
).orderBy("county_name", "name").show(100, truncate=False)

Offshore regions:
+----------+-------------------+-------------------+-------------------+-----------------+----------+---------+
|station_id|name               |county_name        |municipality_name  |municipality_code|latitude  |longitude|
+----------+-------------------+-------------------+-------------------+-----------------+----------+---------+
|SN76956   |GOLIAT FPSO        |BARENTSHAVET       |BARENTSHAVET       |2500             |71.3112   |22.25    |
|SN20924   |HJELMSØYBANKEN II  |BARENTSHAVET       |BARENTSHAVET       |2500             |72.502    |20.1691  |
|SN76909   |JOHAN CASTBERG     |KONTINENTALSOKKELEN|KONTINENTALSOKKELEN|2300             |72.48405  |20.23393 |
|SN20961   |ALVHEIM            |NORDSJØEN          |NORDSJØEN          |2600             |59.5722   |1.9917   |
|SN76959   |BRAGE              |NORDSJØEN          |NORDSJØEN          |2600             |60.5425   |3.0468   |
|SN20954   |DRAUPNER           |NORDSJØEN          |NORDSJØEN          |2600          